# Week 35, Day 1: Receptive fields from natural stimuli: whitening and regularisation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w035/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: STA against whitened STA

For stimulus correlations of 0 and 0.9 (an AR(1) process), simulate an LNP neuron with filter k, and compute the STA (X^T y / sum y) and the whitened STA (least squares). Store their cosine similarities with k in `sims[rho]` as (sta, whitened).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(35)
D, T = 20, 20000
k = np.exp(-np.arange(D) / 4) * np.sin(np.arange(D) / 2.5); k /= np.linalg.norm(k)

def stimulus(rho, n):
    x = np.empty(n); x[0] = r.normal()
    e = r.normal(size=n) * np.sqrt(1 - rho**2)
    for t in range(1, n):
        x[t] = rho * x[t - 1] + e[t]
    return x

def design(x):
    return np.lib.stride_tricks.sliding_window_view(x, D)[: len(x) - D + 1][:, ::-1]   # row t: x[t], x[t-1], ...

cos = lambda v: v @ k / np.linalg.norm(v)
sims = {}
for rho in (0.0, 0.9):
    X = design(stimulus(rho, T + D - 1))
    lin = X @ k
    y = r.poisson(np.exp(lin / lin.std() - 2.5))
    sta = ...
    wsta = ...
    sims[rho] = (cos(sta), cos(wsta))
    print(f"rho = {rho}: {y.sum()} spikes; STA similarity {sims[rho][0]:.3f}, whitened {sims[rho][1]:.3f}")
plt.plot(k, label="true filter"); plt.plot(sta / np.linalg.norm(sta), label="STA (rho 0.9)"); plt.plot(wsta / np.linalg.norm(wsta), label="whitened")
plt.xlabel("lag (bins)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_white():
    "with white noise both estimates align with the filter (similarity above 0.95)"
    assert sims[0.0][0] > 0.95 and sims[0.0][1] > 0.95

def test_corr():
    "with correlated noise the STA's similarity falls below 0.85 and the whitened STA stays above 0.9"
    assert sims[0.9][0] < 0.85 and sims[0.9][1] > 0.9

test_white()
test_corr()
print("All checks passed")

### Exercise: The stimulus covariance spectrum

Compute the eigenvalues of the empirical stimulus covariance X^T X / T for each correlation, and store the ratio of the largest to the smallest in `ratio[rho]`.

In [ ]:
ratio = {}
for rho in (0.0, 0.9):
    X = design(stimulus(rho, T + D - 1))
    ev = ...
    ratio[rho] = ev.max() / ev.min()
    print(f"rho = {rho}: eigenvalues from {ev.min():.3f} to {ev.max():.2f}, ratio {ratio[rho]:.0f}")

In [ ]:
import numpy as np
def test_spectrum():
    "the correlated stimulus's covariance eigenvalues span more than a factor of 100, the white one's less than 2"
    assert ratio[0.9] > 100 and ratio[0.0] < 2

test_spectrum()
print("All checks passed")

### Exercise: Choosing the penalty

With only 2,000 bins of the correlated stimulus, split them into 1,500 for fitting and 500 for validation. For each λ on the grid, fit the ridge estimate on the fitting set and compute the mean squared error of its predictions of the validation counts. Store the errors in `val_err`, and the unregularised and best ridge estimates in `k_ols` and `k_ridge`.

In [ ]:
X = design(stimulus(0.9, 2000 + D - 1))
lin = X @ k
y = r.poisson(np.exp(lin / lin.std() - 1.5))
fit_, val = np.arange(2000) < 1500, np.arange(2000) >= 1500
Xc, yc = X - X[fit_].mean(0), y - y[fit_].mean()

def ridge(lam):
    A = Xc[fit_]
    return ...

lams = [0, 1, 10, 100, 1000, 10000, 100000]
val_err = [np.mean((yc[val] - Xc[val] @ ridge(l)) ** 2) for l in lams]
k_ols, k_ridge = ridge(0), ridge(lams[int(np.argmin(val_err))])
for l, e in zip(lams, val_err):
    print(f"lambda {l:6d}: validation error {e:.4f}")
print(f"similarity to the true filter: unregularised {cos(k_ols):.3f}, ridge {cos(k_ridge):.3f}, STA {cos(X[fit_].T @ y[fit_]):.3f}")

In [ ]:
import numpy as np
def test_cv():
    "the best lambda by validation error lies strictly inside the grid"
    i = int(np.argmin(val_err))
    assert 0 < i < len(lams) - 1

def test_better():
    "the cross-validated ridge estimate is closer to the true filter than the unregularised one"
    assert cos(k_ridge) > cos(k_ols)

test_cv()
test_better()
print("All checks passed")